# Long Context, and Where It Breaks: Lost in the Middle

The full code of [Long Context, and Where It Breaks: Lost in the Middle](https://residual-stream.lvtan71.workers.dev/posts/long-context-lost-in-the-middle/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/long-context-lost-in-the-middle) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/long-context-lost-in-the-middle.ipynb)

**Hardware:** Use a GPU runtime (Colab: Runtime → Change runtime type → T4 GPU). About 80 min end to end.
Downloads Qwen2.5-1.5B-Instruct (about 3 GB).

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
%pip install -q "torch==2.14.*" "transformers==5.17.*" huggingface_hub

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "long-context-lost-in-the-middle"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

## `both_tasks.py`

Runs `position.py kv`, then `position.py needle`.

In [ ]:
!{sys.executable} both_tasks.py

The post printed:

```text
kv     ~  763 prompt tokens: right at depth 0 / 25 / 50 / 75 / 100%: 12/12 12/12 12/12 11/12 12/12   (128 s so far)
kv     ~ 2106 prompt tokens: right at depth 0 / 25 / 50 / 75 / 100%: 12/12 11/12 12/12 12/12 12/12   (340 s so far)
kv     ~ 4106 prompt tokens: right at depth 0 / 25 / 50 / 75 / 100%: 12/12 11/12 10/12  8/12 12/12   (705 s so far)
kv     ~ 8151 prompt tokens: right at depth 0 / 25 / 50 / 75 / 100%: 12/12 10/12  6/12 11/12 12/12   (1411 s so far)
needle ~  822 prompt tokens: right at depth 0 / 25 / 50 / 75 / 100%:  6/6  6/6  6/6  6/6  6/6   (40 s so far)
needle ~ 2072 prompt tokens: right at depth 0 / 25 / 50 / 75 / 100%:  6/6  6/6  6/6  6/6  6/6   (122 s so far)
needle ~ 4072 prompt tokens: right at depth 0 / 25 / 50 / 75 / 100%:  6/6  6/6  6/6  6/6  6/6   (269 s so far)
needle ~ 8072 prompt tokens: right at depth 0 / 25 / 50 / 75 / 100%:  6/6  5/6  6/6  6/6  6/6   (581 s so far)
```

## `more_trials.py`

Runs `position.py kv 120,240 12 -b`: 12 new trials at 8,000 tokens and 12 at 16,000.

In [ ]:
!{sys.executable} more_trials.py

The post printed:

```text
kv     ~ 8142 prompt tokens: right at depth 0 / 25 / 50 / 75 / 100%: 12/12  7/12  9/12  9/12 12/12   (936 s so far)
kv     ~16189 prompt tokens: right at depth 0 / 25 / 50 / 75 / 100%: 12/12  2/12 10/12  5/12 12/12   (2448 s so far)
```

## `key_first.py`

The 12 objects of the 8,100-token, 50%-depth cell, regenerated from the same seed, asked with the key after the data only and with it before and after; every answer classified.

In [ ]:
!{sys.executable} key_first.py

The post printed:

```text
after only: right: 6; none of these: '57615999-1111-2222-3333-444444444444': 1; right value, a character or two wrong: 1; another pair's value (pair 40, target 60): 1; another pair's value (pair 55, target 60): 1; another pair's value (pair 52, target 60): 1; another pair's value (pair 50, target 60): 1
before and after: right: 12
```

## `dilution.py`

Softmax dilution: the attention share of one key that scores a fixed margin above n - 1 equal others.

In [ ]:
!{sys.executable} dilution.py

The post printed:

```text
score margin 4:     100 keys  35.5%,   1,000 keys   5.2%,  10,000 keys   0.5%, 100,000 keys   0.1%
score margin 8:     100 keys  96.8%,   1,000 keys  74.9%,  10,000 keys  23.0%, 100,000 keys   2.9%
margin needed for half the attention among 100,000 keys: ln(99,999) = 11.5
```

## `sinks.py`

Attention weights of a 1,024-token passage: the share on the first token, the last four and the rest, per layer and per head, and which position gets the most attention when the first token changes.

In [ ]:
!{sys.executable} sinks.py

The post printed:

```text
layer  0: first token  0.0%, last 4 tokens 60.4%, the other ~1,000 39.6%
layer  3: first token 35.1%, last 4 tokens 21.5%, the other ~1,000 43.4%
layer  6: first token 49.5%, last 4 tokens 17.3%, the other ~1,000 33.3%
layer  9: first token 31.7%, last 4 tokens 20.2%, the other ~1,000 48.1%
layer 12: first token 34.9%, last 4 tokens 14.3%, the other ~1,000 50.8%
layer 15: first token 16.8%, last 4 tokens 28.7%, the other ~1,000 54.4%
layer 18: first token 20.2%, last 4 tokens 33.0%, the other ~1,000 46.9%
layer 21: first token 30.6%, last 4 tokens 12.1%, the other ~1,000 57.3%
layer 24: first token 46.8%, last 4 tokens 17.3%, the other ~1,000 36.0%
layer 27: first token  0.1%, last 4 tokens 13.0%, the other ~1,000 86.9%
averaged over all 28 layers: first token 30.5%, last 4 tokens 21.1%; a uniform split would give the first token 0.29%
heads in layers 1-26 putting over half their attention on the first token: 48 of 312; under 5%: 19
passage at character 100,000                 starts with 'rive'      : most attention on position 0 ('rive'), 29.0%
passage at character 200,000                 starts with ' then'     : most attention on position 0 (' then'), 30.5%
passage at character 300,000                 starts with 'Parameter' : most attention on position 0 ('Parameter'), 32.3%
passage at character 400,000                 starts with '7'         : most attention on position 1 ('"'), 30.9%
first passage, first token moved to the end  starts with ' climbs'   : most attention on position 0 (' climbs'), 30.4%
first passage with '7'    in front: most attention on position 1 (' then'), 30.4%
first passage with '3'    in front: most attention on position 1 (' then'), 30.4%
first passage with 'A'    in front: most attention on position 0 ('A'), 30.3%
first passage with '.'    in front: most attention on position 0 ('.'), 30.4%
first passage with '\n'   in front: most attention on position 0 ('\n'), 30.4%
first passage with 'the'  in front: most attention on position 0 ('the'), 30.4%
```